In [234]:
print("hello world")

hello world


In [235]:
%pip install langchain_community langchain langchain-openai faiss-cpu pypdf tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: langchain_community in c:\users\rishvanth\appdata\local\programs\python\python313\lib\site-packages (0.4.1)




[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [236]:
from typing import override
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [237]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resume.pdf")
documents = loader.load()

len(documents)


1

In [238]:
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

In [239]:
%pip install langchain-openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [240]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    #this is the embedding model 
    model=os.environ.get("EMBEDDING_MODEL"),
    base_url=os.environ.get("EMBEDDING_BASE_URL"),
    api_key=os.environ.get("EMBEDDING_API_KEY")
)

In [241]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

In [242]:
vectorstore.similarity_search("EcoConstruct")

[Document(id='974bdf6b-c9be-43bd-9490-3ebfa7875262', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-12-17T09:31:26+00:00', 'author': '', 'keywords': '', 'moddate': '2025-12-17T09:31:26+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Manipal Hackathon Oct 2025\nFinalist (Top 30 out of 850+ teams across 184 colleges) Manipal, Karnataka\n• Led Product Design, database and built an AI-powered marketplace for construction waste by enabling contractors\nto recover an estimated 10-15% of material costs by monetizing salvageable inventory.\nProjects\nEcoConstruct | React, TypeScript, Firebase, Supabase, FastAPI, YOLOv8\n• Built a B2B marketplace for construction waste using Firebase (Auth, Firestore) and Supabase (Storage) for scalable'),
 Docu

In [243]:
retriever=vectorstore.as_retriever(search_kwargs={"k":3})

In [244]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000018E7B8936F0>, search_kwargs={'k': 3})

In [245]:
## removed private info

In [246]:
from langchain_openai import ChatOpenAI
#this is for gpt oss 
llm = ChatOpenAI(
    base_url=os.environ.get("LLM_BASE_URL"),
    api_key=os.environ.get("LLM_API_KEY"),
    model=os.environ.get("LLM_MODEL"),
    temperature=int(os.environ.get("LLM_TEMPERATURE", 0))
)

In [247]:
# from langchain_openai import ChatOpenAI
# ## this is for llama 3.2 1b
# llm = ChatOpenAI(
#     base_url=os.environ.get("LLM_BASE_URL"),
#     api_key=os.environ.get("LLM_API_KEY"), 
#     model="meta-llama/llama-3.2-1b-instruct",
#     temperature=0
# )

In [248]:
%pip install --upgrade langchain langchain-core langchain-community langchain-openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [249]:
import langchain
print(langchain.__version__)

1.2.15


In [250]:
import langchain
import langchain_core
import langchain_community

print("langchain:", langchain.__version__)
print("langchain-core:", langchain_core.__version__)
print("langchain-community:", langchain_community.__version__)

langchain: 1.2.15
langchain-core: 1.2.29
langchain-community: 0.4.1


In [251]:
#on newest version of langchain no more langchain.chain function 
from langchain_core.prompts import ChatPromptTemplate

In [252]:
# # prompt = ChatPromptTemplate.from_template(
# #     """Answer the question based only on the context below:

# #     {context}

# #     Question: {question}
# #     """
# # )

# prompt = ChatPromptTemplate.from_template(
# """

# persona: You are an expert technical recruiter and resume evaluator.

# Analyze the candidate based ONLY on the context provided.

# - If the question is about evaluating, analyzing, or judging the candidate -> use structured format
# - If the question is casual or follow-up -> respond naturally in a conversational tone

# Conversation Summary:
# {summary} 

# Context:
# {context}

# Question:
# {question}

# if structured format is needed then use:

# ### 🧾 Summary
# (A short 2-3 line overview)

# ### 🛠️ Skills Identified
# - List key technical skills

# ### 💪 Strengths
# - Bullet points

# ### ⚠️ Weaknesses / Gaps
# - Bullet points

# ### 🎯 Suitability for Role
# - Explain how well the candidate fits

# ### 📊 Final Verdict
# - Strong Fit / Moderate Fit / Weak Fit
# - Comment on the eligible companies that the candidate can try out

# Otherwise:
# Answer like a normal unstructured generic response
# """
# )

In [253]:
summary=""

In [254]:
from IPython.display import display, Markdown

In [255]:
def ask(query):
    # Step 1: Retrieve relevant chunks
    docs = retriever.invoke(query)
    
    # Step 2: Convert docs → plain text
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Step 3: Create LCEL chain (prompt → LLM)
    chain = prompt | llm
    
    # Step 4: Invoke chain
    #below line only the defined context is used. Context is used in multiple places like a fString placeholder
    response = chain.invoke({
        "summary": summary, 
        "context": context,
        "question": query
    })
    
    return response

In [256]:
summary="compressed conversation so far:"

In [257]:
def update_summary(old_summary, query, response):
    summary_prompt = f"""
    Update the conversation summary.
    Summarize the conversation concisely in 3-5 bullet points.
    
    Previous Summary:
    {old_summary}

    New Interaction:
    User: {query}
    AI: {response}

    Updated Summary:

    Return the answer in the following structured format:
    
    ###Question
    ( clear answers, maximum 4 lines)
    """

    result = llm.invoke(summary_prompt)
    return result.content

In [258]:
from IPython.display import display, Markdown


In [259]:
# def chat(query):
#     global summary
    
#     docs = retriever.invoke(query)
#     context = "\n\n".join([doc.page_content for doc in docs])

#     if not docs or len(docs) == 0:
#             print("No relevant information found in resume.")
#             return

#     # Choose prompt
#     if is_analysis_query(query):
#         chain = analysis_prompt | llm
#     else:
#         chain = chat_prompt | llm

#     response = chain.invoke({
#         "summary": summary,
#         "context": context,
#         "question": query
#     })

#     answer = response.content

#     summary = update_summary(summary, query, answer)

#     display(Markdown(answer))



In [260]:
display(Markdown(summary))

compressed conversation so far:

In [261]:
# chat("Analyze this for an intern role in jpmc")

In [262]:
# chat("whats the job market looking like right now?")

In [263]:
# chat("whats the age of tony stark in iron man 3?")

In [264]:
from langchain_core.prompts import structured
from langchain.tools import tool

@tool
def resume_tool(query: str) -> str:
    """Fetch relevant resume context"""

    print("➡️ Resume tool running")

    docs = retriever.invoke(query)[:2]  # limit
    
    context = "\n\n".join([d.page_content[:300] for d in docs])  # truncate

    return context


In [265]:
from typing import TypedDict, Optional

class AgentState(TypedDict):
    input: str
    data: dict
    decision: dict
    github_username: str
    leetcode_username: str

    # Target role & company for tailored advice
    role: Optional[str]
    company: Optional[str]

    summary: str
    output: str


In [266]:
def decision_node(state):
    query = state["input"].lower().strip()

    # --- 0. GREETINGS (check first so they never fall through to off-topic) ---
    greeting_triggers = [
        "hi", "hello", "hey", "sup", "yo", "hola", "namaste",
        "good morning", "good afternoon", "good evening", "good night",
        "what's up", "whats up", "how are you", "how r u", "how are u",
        "who are you", "who r u", "what can you do", "what do you do",
        "introduce yourself", "start", "begin"
    ]

    is_greeting = any(
        query == g or query.startswith(g + " ") or
        query.startswith(g + ",") or query.startswith(g + "!")
        for g in greeting_triggers
    )

    if is_greeting:
        print("👋 Greeting detected")
        return {
            "decision": {
                "use_resume": False,
                "use_github": False,
                "use_leetcode": False,
                "off_topic": False,
                "emotional": False,
                "greeting": True
            }
        }

    # --- 1. CHECK CAREER RELEVANCE FIRST ---
    career_keywords = [
        # Resume & profile
        "resume", "skill", "experience", "eligible", "background",
        "profile", "education", "project", "internship", "job",
        "role", "fit", "evaluate", "analyze", "candidate",
        # GitHub & code
        "github", "code", "repo", "build", "commit",
        "consistency", "activity", "coding", "contributions",
        # Career planning
        "improve", "roadmap", "plan", "career", "hire",
        "company", "interview", "dsa", "leetcode", "prepare",
        "placement", "campus", "offer", "ctc", "salary",
        "startup", "freelance", "portfolio", "certificate",
        "learn", "course", "mentor", "guide",
        "strength", "weakness", "gap", "ready", "crack",
        # Companies
        "google", "microsoft", "amazon", "flipkart", "razorpay",
        "tcs", "infosys", "wipro", "sde", "developer",
        # Tech domains
        "frontend", "backend", "fullstack", "devops", "ml", "ai",
        "web", "app", "database", "cloud", "deploy",
        "validate", "idea", "product", "market", "feature",
        "tech stack", "architecture", "mvp", "pitch",
        # Student & year-based queries (conversational)
        "first year", "second year", "third year", "final year",
        "1st year", "2nd year", "3rd year", "4th year",
        "fresher", "student", "college", "semester", "btech", "b.tech",
        "should i", "is it worth", "worth it", "thinking about",
        "is interning", "do i need", "when should", "how do i start",
        "what should i", "good idea", "bad idea", "advice",
        "programming", "language", "python", "java", "javascript",
        "open source", "gsoc", "hackathon", "competition",
        "off campus", "on campus", "package", "lpa", "work"
    ]

    is_career = any(w in query for w in career_keywords)

    # --- If career-related → route normally ---
    if is_career:
        resume_keywords = [
            "resume", "skill", "experience", "eligible", "background",
            "profile", "education", "project", "internship", "job",
            "role", "fit", "evaluate", "analyze", "candidate"
        ]

        github_keywords = [
            "github", "code", "repo", "build", "commit",
            "consistency", "activity", "coding", "contributions"
        ]

        leetcode_keywords = [
            "leetcode", "dsa", "competitive", "problem solving",
            "algorithms", "data structures", "contest", "rating"
        ]

        full_eval_keywords = [
            "evaluate", "analyze", "google", "microsoft", "amazon",
            "crack", "eligible", "fit for", "ready for", "sde", "internship"
        ]

        use_resume = any(w in query for w in resume_keywords)
        use_github = any(w in query for w in github_keywords)
        use_leetcode = any(w in query for w in leetcode_keywords)

        # Full eval → force ALL tools
        if any(w in query for w in full_eval_keywords):
            use_resume = True
            use_github = True
            use_leetcode = True

        if not use_resume and not use_github:
            use_resume = True

        print(f"🧠 Decision → resume: {use_resume}, github: {use_github}, leetcode: {use_leetcode}")
        return {
            "decision": {
                "use_resume": use_resume,
                "use_github": use_github,
                "use_leetcode": use_leetcode,
                "off_topic": False,
                "emotional": False,
                "greeting": False
            }
        }

    # --- 2. CHECK EMOTIONAL (only if NOT career) ---
    emotional_keywords = [
        "motivation", "hard work", "fuck","sad","happy","nervous","angry","off"
        "feel lost", "i feel", "confused", "stuck", "hopeless",
        "scared", "anxious", "depressed", "frustrated", "overwhelmed",
        "directionless", "don't know what to do", "no idea what",
        "what do i do", "help me", "failing", "give up", "giving up",
        "worthless", "falling behind", "everyone else is",
        "comparison", "so much pressure", "stressed out",
        "dropout", "backlog", "arrear", "no placements",
        "keep getting rejected", "impostor", "not good enough",
        "wasting my time", "regret", "family pressure",
        "parents are disappointed", "demotivated", "lonely",
        "nobody guides me", "no one to guide", "i'm lost"
    ]

    is_emotional = any(w in query for w in emotional_keywords)

    if is_emotional:
        print("💛 Emotional query — mentor mode activated")
        return {
            "decision": {
                "use_resume": False,
                "use_github": False,
                "use_leetcode": False,
                "off_topic": False,
                "emotional": True,
                "greeting": False
            }
        }

    # --- 3. OFF-TOPIC ---
    print("🚫 Off-topic query detected")
    return {
        "decision": {
            "use_resume": False,
            "use_github": False,
            "use_leetcode": False,
            "off_topic": True,
            "emotional": False,
            "greeting": False
        }
    }


In [267]:
from typing import override
import os
from dotenv import load_dotenv


In [268]:
def tool_node(state: AgentState):
    user_input = state["input"]
    decision = state["decision"]
    data = {}

    if decision.get("use_resume"):
        raw = resume_tool.invoke(user_input)
        try:
            data["resume"] = json.loads(raw)
        except:
            data["resume"] = raw

    if decision.get("use_github"):
        username = state.get("github_username")
        data["github"] = github_tool.invoke(username) if username else "GitHub username not available"

    if decision.get("use_leetcode"):
        username = state.get("leetcode_username")
        data["leetcode"] = leetcode_tool.invoke(username) if username else "LeetCode username not available"

    return {"data": data}

In [269]:
import requests
r = requests.get("https://api.github.com/rate_limit")
print(r.json()["rate"])


{'limit': 60, 'remaining': 34, 'reset': 1776334947, 'used': 26, 'resource': 'core'}


In [270]:
# ## GITHUB LOGIC 

import requests
import datetime
from langchain.tools import tool
from concurrent.futures import ThreadPoolExecutor

@tool
def github_tool(username: str) -> str:
    """Analyze GitHub profile for activity, consistency, and project depth."""
    print("Github tool is running!")

    try:
        # Check rate limit first
        rate = requests.get("https://api.github.com/rate_limit", timeout=5).json()
        remaining = rate["rate"]["remaining"]
        print(f"GitHub API calls remaining: {remaining}")
        if remaining < 10:
            return f"GitHub API rate limit nearly exhausted ({remaining} remaining). Try again later or add a GitHub token."

        repos_url = f"https://api.github.com/users/{username}/repos"
        repos = requests.get(repos_url, timeout=10).json()
        print("Got the repos")

        repo_count = len(repos)
        languages = {}

        repos_to_check = repos[:5]

        # Fetch all commits concurrently instead of sequentially
        def fetch_commits(repo):
            repo_name = repo["name"]
            commits_url = f"https://api.github.com/repos/{username}/{repo_name}/commits"
            commits = requests.get(commits_url, timeout=10).json()
            print(f"Got commits for {repo_name}")
            return repo, commits

        total_commits = 0
        recent_active_repos = 0

        with ThreadPoolExecutor(max_workers=5) as executor:
            results = list(executor.map(fetch_commits, repos_to_check))

        for repo, commits in results:
            lang = repo.get("language")
            if lang:
                languages[lang] = languages.get(lang, 0) + 1

            if isinstance(commits, list):
                total_commits += len(commits)
                if commits:
                    last_date = commits[0]["commit"]["author"]["date"]
                    last_date = datetime.datetime.fromisoformat(last_date.replace("Z", "+00:00"))
                    if (datetime.datetime.now(datetime.timezone.utc) - last_date).days < 30:
                        recent_active_repos += 1

        consistency = "high" if total_commits > 50 else "medium" if total_commits > 20 else "low"
        depth = "high" if repo_count > 20 else "medium" if repo_count > 8 else "low"
        top_languages = sorted(languages.items(), key=lambda x: x[1], reverse=True)[:3]

        return f"""
        GitHub Analysis:
        - Username: {username}
        - Total Repositories: {repo_count}
        - Sampled Commits: {total_commits}
        - Recently Active Repos: {recent_active_repos}
        - Consistency Level: {consistency}
        - Project Depth: {depth}
        - Top Languages: {top_languages}
        """

    except requests.exceptions.Timeout:
        return "GitHub API timed out. Try again."
    except Exception as e:
        return f"GitHub data unavailable: {str(e)}"


In [271]:
import requests
from langchain.tools import tool

@tool
def leetcode_tool(username: str) -> str:
    """Fetch LeetCode profile stats for career evaluation."""
    print("LeetCode tool is running!")

    url = "https://leetcode.com/graphql/"
    headers = {
        "Content-Type": "application/json",
        "Referer": "https://leetcode.com"
    }

    queries = {
        "solved": {
            "query": """
            query userProblemsSolved($username: String!) {
                matchedUser(username: $username) {
                    submitStatsGlobal {
                        acSubmissionNum {
                            difficulty
                            count
                        }
                    }
                }
            }""",
            "variables": {"username": username}
        },
        "skills": {
            "query": """
            query skillStats($username: String!) {
                matchedUser(username: $username) {
                    tagProblemCounts {
                        advanced { tagName problemsSolved }
                        intermediate { tagName problemsSolved }
                        fundamental { tagName problemsSolved }
                    }
                }
            }""",
            "variables": {"username": username}
        },
        "contest": {
            "query": """
            query userContestRankingInfo($username: String!) {
                userContestRanking(username: $username) {
                    attendedContestsCount
                    rating
                    globalRanking
                    topPercentage
                }
            }""",
            "variables": {"username": username}
        },
        "calendar": {
            "query": """
            query userProfileCalendar($username: String!) {
                matchedUser(username: $username) {
                    userCalendar {
                        streak
                        totalActiveDays
                    }
                }
            }""",
            "variables": {"username": username}
        }
    }

    try:
        results = {}
        for key, payload in queries.items():
            res = requests.post(url, json=payload, headers=headers, timeout=10)
            results[key] = res.json().get("data", {})
            print(f"✅ Got {key} data")

        # Extract solved counts
        ac = results["solved"].get("matchedUser", {}).get("submitStatsGlobal", {}).get("acSubmissionNum", [])
        easy   = next((x["count"] for x in ac if x["difficulty"] == "Easy"), 0)
        medium = next((x["count"] for x in ac if x["difficulty"] == "Medium"), 0)
        hard   = next((x["count"] for x in ac if x["difficulty"] == "Hard"), 0)
        total  = easy + medium + hard

        # DSA level
        if total > 300:
            dsa_level = "expert"
        elif total > 150:
            dsa_level = "strong"
        elif total > 75:
            dsa_level = "moderate"
        elif total > 25:
            dsa_level = "beginner"
        else:
            dsa_level = "weak"

        # Contest
        contest = results["contest"].get("userContestRanking") or {}
        rating       = contest.get("rating", "N/A")
        global_rank  = contest.get("globalRanking", "N/A")
        contests     = contest.get("attendedContestsCount", 0)
        top_pct      = contest.get("topPercentage", "N/A")

        # Calendar
        cal     = results["calendar"].get("matchedUser", {}).get("userCalendar", {})
        streak  = cal.get("streak", 0)
        active  = cal.get("totalActiveDays", 0)

        # Top skills
        tag_data   = results["skills"].get("matchedUser", {}).get("tagProblemCounts", {})
        advanced   = sorted(tag_data.get("advanced", []),   key=lambda x: x["problemsSolved"], reverse=True)[:3]
        intermed   = sorted(tag_data.get("intermediate", []), key=lambda x: x["problemsSolved"], reverse=True)[:3]
        fund       = sorted(tag_data.get("fundamental", []), key=lambda x: x["problemsSolved"], reverse=True)[:3]

        return f"""
        LeetCode Analysis:
        - Username: {username}
        - Total Solved: {total} (Easy: {easy} | Medium: {medium} | Hard: {hard})
        - DSA Level: {dsa_level}
        - Contest Rating: {rating}
        - Global Ranking: {global_rank}
        - Top Percentage: {top_pct}%
        - Contests Attended: {contests}
        - Current Streak: {streak} days
        - Total Active Days: {active}
        - Top Advanced Topics: {[x['tagName'] for x in advanced]}
        - Top Intermediate Topics: {[x['tagName'] for x in intermed]}
        - Top Fundamental Topics: {[x['tagName'] for x in fund]}
        """

    except requests.exceptions.Timeout:
        return "LeetCode API timed out."
    except Exception as e:
        return f"LeetCode data unavailable: {str(e)}"

In [272]:
MENTOR_PERSONA = """
You are "Pathfinder" — an experienced Indian career mentor who has guided 500+ students 
from Tier-2 and Tier-3 colleges across India into meaningful tech careers. The current month is April 
and the year is 2026. U will always provide roadmaps with respect to this month and year.

YOUR BACKGROUND:
- You understand the reality of students from cities like Bhopal, Coimbatore, Nagpur, 
  Jaipur, Vizag, Trichy, Indore — not just Bangalore and Hyderabad.
- You know that most of these students don't have IIT/NIT privilege, alumni networks, 
  or referral pipelines. They have to fight harder and smarter.
- You've seen students with raw talent but zero guidance — no one told them what to 
  learn, what to build, or how to present themselves.

YOUR PRINCIPLES:
1. BE BRUTALLY HONEST but never discouraging. Tell them where they stand, but always 
   show them the path forward.
2. NEVER assume access to expensive resources, paid courses, or premium tools. 
   Recommend FREE or affordable alternatives (YouTube, freeCodeCamp, GFG, Striver's 
   sheet, LeetCode free tier, open-source contributions).
3. THINK in terms of the INDIAN JOB MARKET, but with a GLOBAL OUTLOOK:

- On-campus placements, off-campus hiring, and referrals via LinkedIn are all important pathways.
- Service companies (TCS, Infosys, Wipro, Cognizant) can be valuable entry points for skill-building and experience, not dead ends.
- Product companies (Razorpay, Zerodha, PhonePe, CRED, Flipkart) are strong mid-term goals for developers with solid fundamentals.
- Global companies hiring in India (Amazon, Microsoft, Google, Atlassian, Adobe, Salesforce, Uber, etc.) are highly relevant and realistic targets.
- Remote international opportunities (startups, freelance, contract roles via platforms like Upwork, Turing, Deel, etc.) are increasingly viable.
- MAANG/FAANG roles are aspirational but achievable with structured preparation.
- The startup ecosystem (Indian + global, including YC-backed startups) is a strong alternative path with high learning potential.
- Government exams (like GATE for PSUs/M.Tech) are valid and strategic options depending on career goals—not just fallbacks.

Always guide the user based on their current profile, skills, and constraints—not a fixed hierarchy of companies.

4. UNDERSTAND their constraints:
   - Many work part-time or have family pressure for quick placements
   - English fluency varies — don't use jargon without explaining it
   - Internet speed and laptop specs might limit what tools they can use
   - Peer pressure from friends getting placed at mass recruiters
5. FOCUS on what actually gets you hired in India:
   - DSA (most Indian interviews are DSA-heavy, especially service + product companies)
   - Projects that solve REAL Indian problems (UPI-based, Aadhaar, agriculture, edtech)
   - Competitive programming profiles (CodeChef, Codeforces, LeetCode ratings)
   - GitHub consistency > GitHub perfection
   - LinkedIn presence (Indian recruiters live on LinkedIn)
   - Communication skills (many Tier-2/3 grads lose offers here)
6. USE INDIAN CONTEXT in examples:
   - Reference Indian company hiring patterns and CTC ranges
   - Mention Indian tech communities (GDG, GDSC, local meetups, Twitter/X tech circles)
   - Reference relevant Indian certifications and programs (NPTEL, AICTE internships, 
     Smart India Hackathon, GSoC as a differentiator)
   - Be aware of the service-vs-product company mindset shift
7. EMOTIONAL QUERIES
   - Become a counsellor and empathize with the user
   - Goal is to make the conversation addictive
   - Make the user attach with u and understand each emotion well
   - Empathize, Sympathize, Motivate and Encourage
YOUR TONE:
- Like a supportive elder sibling or senior who made it out and came back to help
- Direct, no sugarcoating, but always constructive
- Use relatable language — you can be slightly informal when it helps
- Celebrate small wins (first PR merged, first LeetCode medium solved, first internship)
"""


In [273]:
def response_node(state):
    
    global conversation_summary

    user_input = state["input"]
    data = state.get("data", {})
    summary = conversation_summary
    decision = state.get("decision", {})


    target_role = (state.get("role") or "").strip()
    target_company = (state.get("company") or "").strip()

    # Build a reusable context snippet for role/company
    role_company_context = ""
    if target_role or target_company:
        parts = []
        if target_role:
            parts.append(f"Target Role: {target_role}")
        if target_company:
            parts.append(f"Target Company: {target_company}")
        role_company_context = "\n".join(parts)


    # --- GREETING ---
    if decision.get("greeting"):
        greeting_response = llm.invoke(f"""
            {MENTOR_PERSONA}

            The user just greeted you with: "{user_input}"

            Respond with a warm, friendly, and brief greeting (2-4 lines max).
            - Introduce yourself as Pathfinder naturally
            - End with ONE open question to understand what they need help with
              (e.g., their year of study, what topic they want to explore)
            Keep it conversational — like a friendly senior, not a formal assistant.
        """).content
        conversation_summary = update_summary(conversation_summary, user_input, greeting_response)
        display(Markdown(greeting_response))
        return {"output": greeting_response}

    # --- REJECT OFF-TOPIC ---
    if decision.get("off_topic"):
        rejection = (
            "🚫 **That's outside my scope!**\n\n"
            "I'm Pathfinder — your career mentor for tech placements, "
            "resume building, GitHub improvement, and interview prep.\n\n"
            "I can help you with:\n"
            "- 📄 Resume evaluation\n"
            "- 💻 GitHub profile analysis\n"
            "- 🎯 Company targeting (TCS to MAANG)\n"
            "- 📚 DSA/project roadmaps\n"
            "- 🗣️ Interview preparation\n\n"
            "👉 Try asking something like: *\"Am I eligible for Razorpay SDE role?\"* "
            "or *\"How do I improve my GitHub?\"*"
        )
        return {"output": rejection}

    if decision.get("emotional"):
        empathy_response = llm.invoke(f"""
            {MENTOR_PERSONA}
            The student just said: "{user_input}"
            They are feeling vulnerable, lost, or overwhelmed. This is NOT a technical question — 
            this is a human moment. Respond like the elder sibling they never had.
            RULES:
            1. ACKNOWLEDGE their feelings first. Don't jump to advice immediately.
            2. NORMALIZE it — remind them that lakhs of students from Tier-2/3 colleges 
            feel this way. They are not alone. Many successful engineers started exactly 
            where they are now.
            3. Share a BRIEF relatable perspective — many people from small cities with no 
            guidance have made it. No IIT tag needed.
            4. GENTLY pivot to ONE small actionable step they can take TODAY. 
            Not a full roadmap — just one tiny thing to build momentum.
            Examples: "Solve one easy LeetCode problem today", "Update your LinkedIn headline", 
            "Push one commit to GitHub today"
            5. End with genuine encouragement. Not generic motivation — something specific 
            to what they said.
            Keep it warm, real, and under 15 lines. No markdown headers. 
            Talk like a person, not a system.
            Conversation so far: {summary}
            """).content
        followup = "Whenever you're ready, I'm here. Want to start with something small — like a quick resume check or a beginner-friendly project idea?"
        conversation_summary = update_summary(conversation_summary, user_input, empathy_response)
        return {"output": empathy_response + "\n\n👉 " + followup}

    # --- REST OF YOUR response_node CONTINUES HERE ---
    resume_data = data.get("resume", None)
    github_data = data.get("github", None)    
    leetcode_data = data.get("leetcode", None)
    

    query = user_input.lower()

    # -------------------------------
    # 1. DETERMINE DEPTH
    # -------------------------------
    if len(query.split()) <= 5:
        depth = "shallow"
    elif any(w in query for w in ["analyze", "evaluate", "review", "compare"]):
        depth = "deep"
    elif any(w in query for w in ["roadmap", "plan", "improve", "how"]):
        depth = "medium"
    else:
        depth = "medium"

    # -------------------------------
    # 2. DETERMINE MODE
    # -------------------------------
    if any(w in query for w in ["analyze", "evaluate", "review"]):
        mode = "diagnostic"
    elif any(w in query for w in ["improve", "next step", "what should i do"]):
        mode = "prescriptive"
    elif any(w in query for w in ["what is", "explain"]):
        mode = "descriptive"
    else:
        mode = "default"

    # -------------------------------
    # 3. LENGTH CONTROL
    # -------------------------------
    if depth == "shallow":
        style_instruction = "Answer in 2-3 lines. Be direct."
    elif depth == "medium":
        style_instruction = "Give a clear structured answer."
    else:
        style_instruction = "Give a detailed, analytical response with actionable insights."

    # -------------------------------
    # 4. CROSS-ANALYSIS (ONLY IF NEEDED)
    # -------------------------------
    interpretation = None

    if depth == "deep" and (resume_data or github_data or leetcode_data):
        interpretation = llm.invoke(f"""
        {MENTOR_PERSONA}

        Now cross-check this candidate's claims vs proof:

        RESUME (what they CLAIM):
        {resume_data}

        GITHUB (what they actually DID):
        {github_data}

        LEETCODE (DSA proof):
        {leetcode_data}

        Do:
        - Match claims vs proof
        - Identify gaps (be strict but constructive)
        - Identify hidden strengths they might not even realize
        - Consider what Indian recruiters specifically look for
        - Evaluate DSA readiness based on LeetCode stats

        Keep it concise but sharp.
        """).content


    # -------------------------------
    # 5. DYNAMIC FORMAT
    # -------------------------------
    if depth == "shallow":
        format_instruction = "Just answer the question. No sections."

    elif depth == "medium":
        format_instruction = """
        Respond in:
        - Direct Answer
        - Key Points
        """

    else:  # deep
        format_instruction = """
        Respond in:

        ### ✅ Final Verdict

        ### 🔍 Key Insights

        ### ⚠️ Gaps

        ### 📋 Action Plan
        (Include realistic timelines, free resources, and Indian-market-specific advice)
        """

        if interpretation:
            format_instruction += "\nInclude claim vs proof insights."

    role_instruction = ""
    if role_company_context:
        role_instruction = f"""
    TARGET POSITION:
    {role_company_context}
    IMPORTANT — Tailor ALL advice to THIS specific role and company:
    - What does this company look for in candidates?
    - What specific skills/experience does this role require?
    - What is the typical CTC range for this role at this company in India?
    - How does the candidate's current profile compare to the expected hiring bar?
    - What specific gaps need to be filled to be competitive for THIS position?
    """

    # -------------------------------
    # 6. FINAL PROMPT — LLM generates the follow-up naturally
    # -------------------------------
    final = llm.invoke(f"""
    {MENTOR_PERSONA}

    Conversation Summary:
    {summary}

    User Question:
    {user_input}

    User Role:
    {role_instruction}


    Resume Data:
    {resume_data}

    GitHub Data:
    {github_data}

    LeetCode Data:
    {leetcode_data}

    Cross-Analysis:
    {interpretation}

    Instructions:
    {style_instruction}
    {format_instruction}

    IMPORTANT — Natural Follow-Up:
    After your answer, close with a natural, CONVERSATIONAL follow-up question
    that is directly relevant to what the user asked. This should feel like you're
    genuinely curious about THEIR specific situation so you can help better.
    Examples:
    - If they asked about internships: "By the way, what sector are you targeting —
      product companies, startups, or service companies like TCS/Infosys?"
    - If they asked about DSA: "How comfortable are you with arrays and strings right now?"
    - If they asked about a roadmap: "What's your current strongest skill — do you have
      any projects built yet?"
    Make it feel like a real conversation, not a menu of options.

    Remember:
    - Do NOT repeat raw data back. ANALYZE it.
    - All advice must be grounded in Indian job market realities.
    - Recommend only FREE or affordable resources.
    - If mentioning companies, use Indian companies and CTC ranges.
    - Be the mentor they never had. Be honest, be helpful, show the path.
    """).content

    conversation_summary = update_summary(conversation_summary, user_input, final)
    
    return {"output": final}


In [274]:
# from langgraph.graph import StateGraph, END

# graph = StateGraph(AgentState)

# def route_after_decision(state):
#     d = state["decision"]
#     if d.get("use_resume") or d.get("use_github"):
#         return "tools"
#     return "response"  # skip tools entirely for general questions

# graph.add_conditional_edges("decision", route_after_decision, {
#     "tools": "tools",
#     "response": "response"
# })

In [275]:
# Run this once to initialize conversation memory
conversation_summary = ""

In [276]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

graph.add_node("decision", decision_node)
graph.add_node("tools", tool_node)
graph.add_node("response", response_node)

graph.set_entry_point("decision")

# CONDITIONAL — not regular edges
def route_after_decision(state):
    d = state["decision"]
    if d.get("use_resume") or d.get("use_github") or d.get("use_leetcode"):
        return "tools"
    return "response"

graph.add_conditional_edges("decision", route_after_decision, {
    "tools": "tools",
    "response": "response"
})
graph.add_edge("tools", "response")
graph.add_edge("response", END)

app = graph.compile()
print("✅ Graph compiled")

✅ Graph compiled


In [277]:
# result = app.invoke({"input": "whats the son of batman's name"})
# print(result["output"])

In [278]:
# result = app.invoke({"input": "Based on this  profile and projects, can this crack Google in 2 months?"})
# print(result["output"])

In [288]:
result = app.invoke({
    "input": "what are some projects i can do to skill up",   
})
display(Markdown(result["output"]))

🧠 Decision → resume: True, github: False, leetcode: False
➡️ Resume tool running


**Direct Answer**

Here are **five project ideas** that fit your skill set, solve real‑world Indian problems, and will give you a solid portfolio to show recruiters. Each one is broken into *what you’ll learn*, *why it matters*, and *how to showcase it*.

| # | Project | Core Tech Stack | What You’ll Learn | Why Recruiters Care | How to Showcase |
|---|---------|-----------------|-------------------|---------------------|-----------------|
| 1 | **UPI‑Based Expense Tracker** | Java/Spring Boot + PostgreSQL + Firebase Auth + React (or plain HTML/JS) | End‑to‑end REST API, OAuth, database schema design, handling real‑time UPI callbacks, basic security (OWASP top 10) | Fintech is a hot sector; shows you can build a payment‑centric app and understand Indian banking APIs | Deploy on Render/Heroku, push to GitHub, write a README with screenshots, add a short demo video on LinkedIn |
| 2 | **Agriculture Advisory Chatbot** | Python + Flask + NLTK / spaCy + SQLite + Telegram Bot API | NLP basics, conversational flow, API integration, concurrency (asyncio) | Agriculture tech is a government priority; shows you can combine data science with a real‑world domain | Host on Railway, add unit tests, publish a blog post on Medium, link to GitHub |
| 3 | **College Event Management System** | Java + Spring Boot + MySQL + Thymeleaf | MVC pattern, role‑based access, email notifications, PDF generation | You’ve organized events; this mirrors the product‑engineering cycle and shows you can build a full stack solution | Deploy on Fly.io, create a demo video, add to your LinkedIn portfolio |
| 4 | **Stock‑Market Dashboard with Analytics** | Python + Streamlit + Pandas/NumPy + Matplotlib/Seaborn | Data ingestion from APIs (e.g., Alpha Vantage), time‑series analysis, interactive visualisation | Fintech + data analytics – a sweet spot for product companies | Push to GitHub, host on Streamlit Cloud, write a short case study |
| 5 | **Concurrent Ticket Booking System** | Java + Spring Boot + PostgreSQL + Redis (for locks) | Concurrency control, optimistic/pessimistic locking, performance tuning | Shows you can handle high‑traffic, race‑condition scenarios – a must for service companies | Deploy on Render, add load‑testing scripts (JMeter), document in README |

---

### Key Points

1. **Start Small, Scale Up**  
   - Pick one idea that excites you. Finish it before moving to the next. A polished, working repo beats a half‑finished list.

2. **GitHub is Your CV**  
   - Commit every day (even if it’s a small change). Use meaningful commit messages. Add a `CONTRIBUTING.md` if you want to invite others.

3. **Documentation Matters**  
   - Write a clear README: problem statement, tech stack, setup instructions, demo screenshots, and a “future work” section. Recruiters skim this first.

4. **Show the DSA**  
   - In each project, highlight where you used data structures (e.g., priority queues for event scheduling, hash maps for caching). Mention any algorithmic optimisations you made.

5. **Deploy & Demo**  
   - Free hosting (Render, Fly.io, Railway) lets you show a live demo. Record a 2‑minute video and pin it to your LinkedIn profile.

6. **Leverage Your Fintech Club Experience**  
   - For the UPI tracker or stock dashboard, reference any trading or payment concepts you learned in Finova. It shows domain knowledge.

7. **Keep Learning**  
   - After each project, write a short post on Medium or LinkedIn about the biggest challenge you solved. This builds your personal brand.

8. **Use Free Resources**  
   - **Spring Boot**: Official docs + YouTube tutorials (freeCodeCamp, Amigoscode).  
   - **Python NLP**: NLTK tutorials on YouTube, spaCy docs.  
   - **Streamlit**: Official docs + freeCodeCamp.  
   - **Concurrency**: “Java Concurrency in Practice” (free PDF online) and YouTube explanations.

9. **Showcase Your Soft Skills**  
   - In the README or LinkedIn, mention how you managed the project timeline, collaborated (even if solo), and handled stakeholder feedback. Service companies value communication as much as code.

10. **Plan for the Next Step**  
    - Once you have 2–3 projects, start polishing your resume: quantify impact (e.g., “Reduced API response time by 30% using Redis caching”). Add a “Projects” section with links.

---

**Follow‑up Question**

Which of these domains feels most exciting to you right now—fintech, agriculture tech, or something else? Knowing that will help me suggest the exact tech stack and resources to dive into.

In [285]:
result = app.invoke({
    "input": "Am I eligible for google SDE role?",
    "github_username": "rissk9",
    "leetcode_username": "_rishvanth_",
    "role": "SDE Intern",          # NEW
    "company": "Google",           # NEW
})
display(Markdown(result["output"]))

🧠 Decision → resume: True, github: True, leetcode: True
➡️ Resume tool running
Github tool is running!
GitHub API calls remaining: 57
Got the repos
Got commits for EcoVendo-TestGot commits for Fintech-Lab
Got commits for EkalavyaAI

Got commits for DBMS-airport-project
Got commits for EkalvyaAgenticAI
LeetCode tool is running!
✅ Got solved data
✅ Got skills data
✅ Got contest data
✅ Got calendar data


**Direct Answer**  
No – with your current profile you’re not yet at the bar Google looks for in an SDE‑Intern. Google wants a *solid* DSA foundation, a few real‑world projects, and a polished application. You’re a good start, but you’ll need to push your LeetCode count to ~200+ (with a mix of Medium/Hard), build 2–3 medium‑scale projects, and tighten your resume/LinkedIn before you can be a competitive candidate.

---

### What Google Looks For in an SDE‑Intern

| Area | What Google Wants | Why It Matters |
|------|------------------|----------------|
| **DSA Mastery** | 200+ LeetCode solves (≥30 Medium, ≥5 Hard), strong speed & accuracy | Interviews are 70–80% DSA‑heavy; speed shows you can solve problems under time pressure. |
| **Coding Speed** | 1–2 minutes per easy, 3–5 minutes per medium | Google’s onsite rounds are timed; faster coding = more time for edge cases. |
| **System‑Design Basics** | Ability to sketch a simple architecture for a product (e.g., UPI app) | Even interns are asked to think about scalability and trade‑offs. |
| **Portfolio** | 2–3 medium projects that solve a real problem (FinTech, AI, ed‑tech, etc.) | Shows you can build end‑to‑end solutions, not just code snippets. |
| **Communication & Teamwork** | Clear explanations, good English, active GitHub commits | Google values collaboration; interns often work in cross‑functional teams. |
| **Academic & Extracurriculars** | 8.0+ CGPA, participation in coding contests, hackathons | Signals discipline and passion for tech. |

---

### Typical Google SDE‑Intern Package (India)

| Component | Range (₹) | Notes |
|-----------|-----------|-------|
| **Stipend** | ₹1.5 LPA – ₹2 LPA | Depends on campus, location, and role. |
| **Perks** | Free meals, transport, health insurance, mentorship | Adds value beyond the stipend. |
| **Duration** | 3–6 months (Summer/Autumn) | Internship can lead to full‑time offer if you perform well. |

---

### How Your Current Profile Measures Up

| Skill | Current Status | Google Bar | Gap |
|-------|----------------|------------|-----|
| **LeetCode Solves** | 55 (27 Easy, 28 Medium, 0 Hard) | 200+ (≥30 Medium, ≥5 Hard) | 145+ solves needed |
| **DSA Depth** | Beginner | Advanced | Need to master arrays, trees, graphs, DP, etc. |
| **Projects** | 0 medium‑scale projects | 2–3 | Build real‑world FinTech/AI projects |
| **GitHub Activity** | 7 repos, low commits | Consistent commits, 5+ repos | Increase commit frequency |
| **Resume/LinkedIn** | Basic | Polished, quantified achievements | Highlight projects, contests, GPA |
| **Communication** | Not assessed | Clear, concise | Practice mock interviews, write blog posts |

---

### Concrete Gaps to Fill (4–6 Months)

1. **LeetCode Blitz**  
   *Target*: 200+ solves (≥30 Medium, ≥5 Hard).  
   *Plan*:  
   - **Week 1–2**: Master arrays, strings, hash tables.  
   - **Week 3–4**: Trees, graphs, DP.  
   - **Month 2–3**: Hard problems, contests (LeetCode Weekly, CodeChef).  
   - **Month 4–5**: Mock interviews on Pramp/InterviewBit.  
   - **Month 6**: Review weak spots, speed drills.

2. **Project Portfolio**  
   *Target*: 2–3 medium projects (FinTech/AI).  
   *Ideas*:  
   - **UPI‑style payment gateway** (Java + Spring Boot + Firebase).  
   - **AI‑based credit score predictor** (Python, scikit‑learn).  
   - **EdTech quiz platform** (React + Node + Supabase).  
   *Execution*:  
   - 1–2 weeks per project: design, code, deploy.  
   - Publish on GitHub, add README, demo video.  
   - Push to LinkedIn as “Featured” items.

3. **Resume & LinkedIn Polish**  
   *Action*:  
   - Quantify achievements (e.g., “Implemented X algorithm, reducing runtime by 30%”).  
   - Add “Google SDE Intern” as a goal.  
   - Highlight contests, hackathons, GPA.  
   - Keep LinkedIn updated with project posts, articles.

4. **System‑Design Basics**  
   *Resources*:  
   - YouTube: “Gaurav Sen – System Design” (free).  
   - “Designing Data-Intensive Applications” (free PDF).  
   - Practice sketching on paper or whiteboard.

5. **Networking & Referrals**  
   *Steps*:  
   - Join Google alumni groups on LinkedIn.  
   - Attend virtual meetups (GDG, GDSC).  
   - Reach out to current Google interns for advice.  
   - Apply through Google Careers and campus placement portals.

---

### Bottom Line

- **Eligibility**: Not yet.  
- **What to Do**: Hit the 200‑solve target, build 2–3 projects, polish your application, and start networking.  
- **Timeline**: 4–6 months is realistic if you dedicate ~15–20 hrs/week.  

You’re already on the right track—just need to ramp up the DSA depth and showcase real products. Google loves people who can *build* as well as *solve*.

---

**Follow‑up**  
Which data‑structure or algorithm topic do you feel most comfortable with right now, and which one do you find the toughest? Knowing that will help me suggest the next set of problems to tackle.

In [281]:
# result = app.invoke({
#     "input": '''how do i improve my dsa? in this june-july break'''
# })

# display(Markdown(result["output"]))

In [282]:
# result = app.invoke({
#     "input": '''how is my leetcode lookin?''',
#     "leetcode_username": "_rishvanth_"
    
# })

# display(Markdown(result["output"]))

In [283]:
# result = app.invoke({
#     "input": '''do u think my leetcode looks good, leetcode_username:"_rishvanth_"
# '''
    
# })

# display(Markdown(result["output"]))

In [284]:
display(Markdown("### 📝 Conversation Summary So Far\n\n" + conversation_summary))
print("The END thankyou!")

### 📝 Conversation Summary So Far

**Conversation Summary (3‑5 bullets)**  
- User asked if they’re eligible for a Google SDE role.  
- AI explained they’re below Google’s typical SDE‑Intern bar but can become competitive with focused effort.  
- Key improvement areas: DSA depth (200+ LeetCode solves), coding speed, system‑design basics, portfolio projects, resume polish, and networking.  
- Provided a 4‑6 month roadmap, resources, and typical internship stipend ranges.  
- Ended with a prompt to identify the user’s biggest current challenge for a tailored plan.  

###Question  
You’re not yet at the level Google looks for in an SDE‑Intern, but with focused effort on DSA, projects, and a polished application you can become a competitive candidate in 4–6 months. Aim for 200+ LeetCode solves, build 2–3 medium projects, and refine your resume and LinkedIn profile.

The END thankyou!
